# Appendix E, part 1: frequency reconstruction and probe sweep over the fifteen geometries

This notebook produces **three pages**, five geometries per page. Each row is one geometry: on the
left the reconstruction curve, in red, which plots the dominant frequency of the model's output
against the frequency supplied as context; on the right the probe sweep of the *same* geometry, one
cell per swept frequency, coloured by the accuracy of a linear probe on the internal state.

The measurement is the one `reconstruction_figures.ipynb` performs, and nothing here changes it.
When that notebook has already been run its stored tables are reused; otherwise each geometry is
measured, saved, and drawn. The two readings answer different questions and are meant to be read
together, so they share a row: a frequency off the diagonal on the left but still green on the
right lost the tone at the projection rather than at the tokenisation.

Everything this notebook depends on lives in `chronos/bayesian/`.


## 0, Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Where the repository is. Locally the notebook sits inside it and nothing has to be set; on
# Colab the working directory is /content, so the search widens to the usual places and, failing
# those, the repository is cloned. Setting REPO_DIR (or the PATCHALIASING_REPO environment
# variable) to the checkout skips the search entirely.
REPO_DIR = None                    # e.g. "/content/drive/MyDrive/patchAliasing"
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
CLONE_IF_MISSING = True            # False to fail with a message instead of cloning

MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def _ok(p) -> bool:
    return p is not None and (Path(p) / MARKER).exists()


def find_repo() -> Path:
    """Locate the checkout: an explicit setting, then the parents, then the usual Colab places."""
    explicit = REPO_DIR or os.environ.get("PATCHALIASING_REPO")
    if explicit:
        if _ok(explicit):
            return Path(explicit).resolve()
        raise FileNotFoundError(f"REPO_DIR is set to {explicit}, but {MARKER} is not under it")

    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:                       # the notebook inside the checkout
        if _ok(cand):
            return cand

    roots = [here, Path("/content"), Path("/content/drive/MyDrive"),
             Path("/content/drive/MyDrive/Colab Notebooks"), Path.home()]
    for root in roots:                                       # a checkout beside the notebook
        if not root.exists():
            continue
        for cand in [root, *(d for d in root.iterdir() if d.is_dir())]:
            if _ok(cand):
                return cand.resolve()

    if not CLONE_IF_MISSING:
        raise FileNotFoundError(
            f"{MARKER} not found. Set REPO_DIR to the checkout, or allow CLONE_IF_MISSING.")

    target = here / "patchAliasing"                          # last resort: fetch it
    if not (target / ".git").exists():
        print(f"cloning {REPO_URL} -> {target}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not _ok(target):
        raise FileNotFoundError(f"{MARKER} missing from {target}")
    return target.resolve()


REPO  = find_repo()
BAYES = REPO / "chronos" / "bayesian"
sys.path.insert(0, str(BAYES))
sys.modules.pop("probe_lib", None)          # so a git pull is picked up without a kernel restart
print("repository:", REPO)

repository: /content/patchAliasing


In [ ]:
# Dependencies.  A local machine that has already run the Bayesian notebooks has all of these;
# a fresh Colab runtime has numpy, pandas, matplotlib, scikit-learn and torch, but not the
# Chronos pipeline classes, which come from `chronos-forecasting`.
import importlib.util, subprocess, sys

for module, package in (("torch", "torch"), ("chronos", "chronos-forecasting")):
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import chronos
# The repository has its own top-level `chronos/` directory.  It shadows the library whenever the
# working directory is the repository root, and the error that follows is confusing, so it is
# caught here instead.
if not hasattr(chronos, "BaseChronosPipeline"):
    raise ImportError(
        f"`chronos` resolved to {getattr(chronos, '__file__', '?')}, which is the repository's own "
        "folder rather than the chronos-forecasting library. Run this notebook from a working "
        "directory that is not the repository root.")
print("chronos:", chronos.__file__)

chronos: /usr/local/lib/python3.13/dist-packages/chronos/__init__.py


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
import probe_lib as pl

SEED = 42
np.random.seed(SEED)
FS, CTX, PRED, BAND = pl.FS, pl.CTX, pl.PRED, pl.BAND

from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize

# ------------------------------------------------------------------------------------- #
#  CONFIGURATION
# ------------------------------------------------------------------------------------- #
# The fifteen geometries of the deliverable's Table 4, in the order they are reported there.
CLEAN_15 = [(8, 8),
            (16, 8), (16, 12), (16, 16),
            (24, 8), (24, 12), (24, 16), (24, 20), (24, 24),
            (32, 8), (32, 12), (32, 16), (32, 20), (32, 24), (32, 32)]

ROWS_PER_PAGE = 5          # five per page, so three pages
RIGHT_PANEL   = "all"      # "all": the eight probe tasks; "local": the f +/- 1 Hz strip alone

FREQ_STEP   = 1            # sweep step [Hz]
N_PHASE     = 10           # phases per frequency, probing dataset
N_PHASE_GEN = 4            # phases per frequency, rollout
GEN_LEN     = 512          # rollout length, so the output DFT has 1 Hz bins
BATCH       = 64
REUSE_STORED = True        # reuse reconstruction_figures.ipynb's tables when they exist

USE_STUB_FORECASTER = False   # True only to check the layout without the checkpoints

OUT    = BAYES / "_run" / "appendixE"
STORED = BAYES / "_run" / "full" / "pagani"      # where reconstruction_figures.ipynb writes
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "data").mkdir(parents=True, exist_ok=True)

FREQS = np.arange(BAND[0], BAND[1] + 1e-9, FREQ_STEP)
n_pages = int(np.ceil(len(CLEAN_15) / ROWS_PER_PAGE))
print(f"fs={FS} Hz  band={BAND}  {len(FREQS)} frequencies")
print(f"{len(CLEAN_15)} geometries, {ROWS_PER_PAGE} per page -> {n_pages} pages")
print(f"figures -> {OUT / 'figures'}")

fs=512 Hz  band=(2.0, 250.0)  249 frequencies
15 geometries, 5 per page -> 3 pages
figures -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures


## 1, Measurement

The two readings, taken exactly as in `reconstruction_figures.ipynb`.

In [ ]:
class StubProbe:
    """A stand-in for `pl.Probe` that needs no checkpoint: layout checks only.

    It returns a smoothed continuation of the context, which is not a forecast and must never be
    read as one. Every figure produced while `USE_STUB_FORECASTER` is true carries a stamp.
    """
    def __init__(self, P, S):
        self.P, self.S = P, S
        self.tag, self.label = pl.model_tag(P, S), f"stub p{P}-s{S}"
        self.stages = ["output_head"]

    def forecast(self, contexts):
        c = np.asarray(contexts, dtype=np.float32)
        k = np.ones(9) / 9.0
        sm = np.stack([np.convolve(row, k, mode="same") for row in c])
        return np.repeat(sm[:, -1:], PRED, axis=1) * 0.6 + sm[:, -PRED:] * 0.4

    def capture_reg(self, contexts, pipe=None):
        c = np.asarray(contexts, dtype=np.float32)
        return {"output_head": np.stack([c[:, :16], c[:, -16:]], axis=1).reshape(len(c), -1)}

    def close(self):
        pass


def open_probe(P, S, batch_size=64):
    """The real probe, or the stub when the checkpoints are not available."""
    return StubProbe(P, S) if USE_STUB_FORECASTER else pl.Probe(P, S, batch_size=batch_size)


def stamp_stub(fig):
    if USE_STUB_FORECASTER:
        fig.text(0.5, 0.5, "STUB FORECASTER\nNOT A MEASUREMENT", fontsize=42, color="red",
                 alpha=0.16, ha="center", va="center", rotation=30, zorder=99)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold, StratifiedGroupKFold

STAGE = "output_head"      # the projected quantile head
DELTA_BINS = 1             # the local task compares neighbours one step apart


def pagani_context(S: int) -> int:
    """About 512 samples, snapped down to a whole number of strides."""
    return (512 // S) * S


def probe_accuracy(X, y, groups=None, seed=SEED):
    """Cross-validated per-example correctness of a linear probe.

    Band tasks are grouped by frequency, so a probe cannot score by memorising individual
    frequencies; the local task cannot be grouped that way, because its two classes are two
    frequencies, and holds out phases instead.
    """
    if len(np.unique(y)) < 2:
        return None
    if groups is None:
        n_splits = int(min(5, np.bincount(y).min()))
        cv = StratifiedKFold(n_splits, shuffle=True, random_state=seed) if n_splits >= 2 else None
    else:
        n0 = len(np.unique(groups[y == 0])); n1 = len(np.unique(groups[y == 1]))
        n_splits = int(min(5, n0, n1))
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                  random_state=seed) if n_splits >= 2 else None
    if cv is None:
        return None
    train_min = len(y) - int(np.ceil(len(y) / n_splits))
    n_comp = max(2, min(20, X.shape[1], train_min - 1))
    clf = make_pipeline(StandardScaler(), PCA(n_components=n_comp),
                        LogisticRegression(max_iter=2000))
    return (cross_val_predict(clf, X, y, cv=cv, groups=groups) == y).astype(float)


def rollout_dominant(probe, freqs, n_phase, ctx_len):
    """Dominant output frequency after an autoregressive rollout, averaged over phases.

    All (frequency, phase) pairs advance in lockstep, so one rollout step is one batched pass.
    """
    items = [(f, ph) for f in freqs for ph in pl.phases_Sf(f, n_phase)]
    ctx = np.stack([pl.build_context(None, f, ph, ctx_len) for f, ph in items])
    gen = np.zeros((len(items), 0), dtype=np.float32)
    while gen.shape[1] < GEN_LEN:
        step = probe.forecast(ctx)
        gen = np.concatenate([gen, step], axis=1)
        ctx = np.concatenate([ctx, step], axis=1)[:, -ctx_len:]
    gen = gen[:, :GEN_LEN]
    x = gen - gen.mean(axis=1, keepdims=True)          # remove DC before reading the peak
    mag = np.abs(np.fft.rfft(x, axis=1))
    fr = np.fft.rfftfreq(GEN_LEN, d=1 / FS)
    dom = fr[np.argmax(mag[:, 1:], axis=1) + 1]        # and skip the DC bin outright
    out = pd.DataFrame(items, columns=["freq", "phase"])
    out["dominant"] = dom
    return out.groupby("freq")["dominant"].agg(["mean", "std"]).reset_index()


def spectral_accuracy(probe, freqs, n_phase, ctx_len):
    """Per-frequency probe accuracy: the seven band tasks, plus the local f +/- delta task."""
    items = [(f, ph) for f in freqs for ph in pl.phases_Sf(f, n_phase)]
    ctx = np.stack([pl.build_context(None, f, ph, ctx_len) for f, ph in items])
    feats = probe.capture_reg(ctx)[STAGE]
    lab_f = np.array([f for f, _ in items])

    rows, labels = [], []
    for (name, lo, hi, bnd) in pl.BAND_TASKS:
        m = (lab_f >= lo) & (lab_f <= hi)
        fm = lab_f[m]
        ok = probe_accuracy(feats[m], (fm > bnd).astype(int), groups=fm)
        a = np.full(len(freqs), np.nan)
        if ok is not None:
            for ci, f in enumerate(freqs):
                sel = fm == f
                if sel.any():
                    a[ci] = ok[sel].mean()
        rows.append(a); labels.append(f"band task {name}")

    local = np.full(len(freqs), np.nan)
    for ci in range(DELTA_BINS, len(freqs) - DELTA_BINS):
        m = np.isin(lab_f, [freqs[ci - DELTA_BINS], freqs[ci + DELTA_BINS]])
        ok = probe_accuracy(feats[m], (lab_f[m] == freqs[ci + DELTA_BINS]).astype(int))
        if ok is not None:
            local[ci] = ok.mean()
    rows.append(local); labels.append(r"local $f\pm1$ Hz")
    return np.vstack(rows), labels

In [ ]:
def measure_or_load(P, S):
    """(reconstruction table, accuracy matrix, row labels) for one geometry."""
    tag = pl.model_tag(P, S)
    f_rec, f_acc = STORED / f"recon_{tag}.parquet", STORED / f"spectral_accuracy_{tag}.parquet"
    if REUSE_STORED and f_rec.exists() and f_acc.exists():
        acc_df = pd.read_parquet(f_acc)
        print(f"  {tag}: reused")
        return pd.read_parquet(f_rec), acc_df.to_numpy().T, list(acc_df.columns)

    ctx_len = pagani_context(S)
    probe = open_probe(P, S, batch_size=BATCH)
    try:
        rec = rollout_dominant(probe, FREQS, N_PHASE_GEN, ctx_len)
        acc, labels = spectral_accuracy(probe, FREQS, N_PHASE, ctx_len)
    finally:
        probe.close()
    rec.to_parquet(OUT / "data" / f"recon_{tag}.parquet", index=False)
    pd.DataFrame(acc.T, columns=labels, index=FREQS).rename_axis("freq_hz") \
      .to_parquet(OUT / "data" / f"spectral_accuracy_{tag}.parquet")
    print(f"  {tag}: measured (context {ctx_len})")
    return rec, acc, labels


MEASURED = {}
for (P, S) in CLEAN_15:
    MEASURED[(P, S)] = measure_or_load(P, S)
print(f"ready: {len(MEASURED)} geometries")

config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

p8-s8-seed42/model.safetensors: reconstructing file:   0%|          |  0.00B / 34.5MB            

p8-s8-seed42/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

  p8-s8: measured (context 512)


...

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

p24-s20-seed42/model.safetensors: reconstructing file:   0%|          |  0.00B / 34.7MB            

p24-s20-seed42/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

: 

## 2, The three pages

In [ ]:
# The recovery colour scale, and the lock markers, as in reconstruction_figures.ipynb: 0.5 is
# chance on a two-class task, 0.6 was the old "lost" boundary and 0.9 the old "recovered" one.
ACC_MIN = 0.5
ACC_STOPS = [(0.50, "#8c1010"), (0.60, "#d62728"), (0.75, "#f2c14e"),
             (0.90, "#8cc63f"), (1.00, "#1a7a1a")]
_opaque = LinearSegmentedColormap.from_list(
    "recovery_opaque", [((v - ACC_MIN) / (1.0 - ACC_MIN), c) for v, c in ACC_STOPS])
_rgba = _opaque(np.linspace(0, 1, 256)); _rgba[:, 3] = 0.70
CMAP = ListedColormap(_rgba, name="recovery")
CMAP.set_bad((0.72, 0.72, 0.72, 0.70))                      # not covered by any task
CMAP.set_under(tuple(_opaque(0.0)[:3]) + (0.70,))           # below chance
NORM = Normalize(vmin=ACC_MIN, vmax=1.0)
C_RECON = "#c81e3c"
DASH = (0, (4, 3))


def own_locks(P, S):
    """The predicted sites of one geometry: F_lock = {k fs/P} union {c fs/S}."""
    return sorted({round(f, 3) for f in pl.patch_nulls(P)} |
                  {round(f, 3) for f in pl.stride_locks(S)})


def draw_reconstruction(ax, rec, P, S, show_xlabel):
    for f in own_locks(P, S):
        ax.axvline(f, color="k", ls=DASH, lw=0.8, alpha=0.75, zorder=1)
    ax.plot(BAND, BAND, ls=":", color="0.45", lw=1.0, zorder=2)
    std = rec["std"].fillna(0)
    ax.fill_between(rec["freq"], rec["mean"] - std, rec["mean"] + std,
                    color=C_RECON, alpha=0.22, lw=0, zorder=3)
    ax.plot(rec["freq"], rec["mean"], color=C_RECON, lw=1.4, zorder=4)
    ax.set_xlim(*BAND); ax.set_ylim(0, BAND[1])
    ax.tick_params(labelsize=7)
    ax.set_ylabel(f"p{P}-s{S}\ndominant out [Hz]", fontsize=8)
    if show_xlabel:
        ax.set_xlabel("input frequency [Hz]", fontsize=8)
    else:
        ax.set_xticklabels([])


def draw_sweep(ax, acc, labels, P, S, show_xlabel, rows="all"):
    a = np.ma.masked_invalid(np.asarray(acc, float))
    if rows == "local":
        a, labels = a[-1:, :], labels[-1:]
    mesh = ax.pcolormesh(np.arange(len(FREQS) + 1), np.arange(a.shape[0] + 1),
                         a, cmap=CMAP, norm=NORM, shading="flat")
    for f in own_locks(P, S):
        ax.axvline(np.interp(f, FREQS, np.arange(len(FREQS))) + 0.5,
                   color="k", ls=DASH, lw=0.8, alpha=0.9)
    ax.set_yticks(np.arange(a.shape[0]) + 0.5)
    ax.set_yticklabels([l.replace("band task ", "") for l in labels], fontsize=6)
    ax.invert_yaxis()
    ticks = [2, 50, 100, 150, 200, 250]
    ax.set_xticks([np.interp(t, FREQS, np.arange(len(FREQS))) + 0.5 for t in ticks])
    ax.set_xticklabels([str(t) for t in ticks] if show_xlabel else [], fontsize=7)
    if show_xlabel:
        ax.set_xlabel("frequency [Hz]", fontsize=8)
    return mesh

In [ ]:
def build_pages():
    pages = [CLEAN_15[i:i + ROWS_PER_PAGE] for i in range(0, len(CLEAN_15), ROWS_PER_PAGE)]
    written = []
    for pi, page in enumerate(pages, start=1):
        fig = plt.figure(figsize=(11.0, 13.0))
        gs = GridSpec(len(page), 2, figure=fig, width_ratios=[1.0, 1.25],
                      hspace=0.18, wspace=0.16,
                      left=0.075, right=0.955, top=0.955, bottom=0.055)
        mesh = None
        for ri, (P, S) in enumerate(page):
            rec, acc, labels = MEASURED[(P, S)]
            last = ri == len(page) - 1
            draw_reconstruction(fig.add_subplot(gs[ri, 0]), rec, P, S, last)
            mesh = draw_sweep(fig.add_subplot(gs[ri, 1]), acc, labels, P, S, last, RIGHT_PANEL)
        fig.suptitle(f"Reconstruction and probe sweep, page {pi} of {len(pages)}"
                     f"   (p{page[0][0]}-s{page[0][1]} to p{page[-1][0]}-s{page[-1][1]})",
                     fontsize=11)
        cb = fig.colorbar(mesh, ax=fig.axes, fraction=0.016, pad=0.012, extend="min")
        cb.set_ticks([0.5, 0.6, 0.75, 0.9, 1.0])
        cb.set_ticklabels(["0.50 chance", "0.60 lost", "0.75", "0.90 recovered", "1.00"])
        cb.ax.tick_params(labelsize=7); cb.set_label("probe accuracy", fontsize=8)
        fig.legend(handles=[Line2D([0], [0], color=C_RECON, lw=1.6,
                                   label="dominant output frequency"),
                            Line2D([0], [0], color="0.45", ls=":", lw=1.2, label="output = input"),
                            Line2D([0], [0], color="k", ls=DASH, lw=1.0,
                                   label=r"predicted lock $f\in\mathcal{F}_{lock}$")],
                   loc="lower center", ncol=3, fontsize=8, frameon=False,
                   bbox_to_anchor=(0.5, 0.012))
        stamp_stub(fig)
        p = OUT / "figures" / f"FIG_E1_page{pi}.png"
        fig.savefig(p, dpi=200, bbox_inches="tight")
        fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
        plt.close(fig)
        written.append(p); print("wrote", p.name)
    return written


PAGES = build_pages()

In [ ]:
# Saving the figures somewhere that outlives the runtime.  On Colab `_run/` lives on the VM's own
# disk and disappears when the session ends, so the figures are mirrored to Drive; elsewhere the
# same call copies them to DRIVE_DIR if that path exists, and is otherwise a no-op.
import importlib.util, shutil

DRIVE_COPY = True
DRIVE_DIR  = "/content/drive/MyDrive/appendixE_figures"


def mirror_to_drive(src=None, dest=DRIVE_DIR, patterns=("*.png", "*.pdf", "*.csv")):
    """Copy the figures out of `_run/` and into Drive.  Returns the destination, or None."""
    if not DRIVE_COPY:
        print("DRIVE_COPY is off; figures stay under", OUT)
        return None
    src = Path(src or (OUT / "figures"))
    on_colab = importlib.util.find_spec("google.colab") is not None
    if on_colab and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")                 # asks once per runtime
    dest = Path(dest)
    if not on_colab and not dest.parent.exists():
        print(f"not on Colab and {dest.parent} does not exist; figures stay under {src}")
        return None
    dest.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in patterns:
        for f in sorted(list(src.glob(pattern)) + list(src.parent.glob(pattern))):
            shutil.copy2(f, dest / f.name)
            copied.append(f.name)
    print(f"copied {len(copied)} files -> {dest}")
    for name in copied:
        print("   ", name)
    return dest


mirror_to_drive()

## 3, What goes in the appendix

One `\includegraphics` per page, at `width=\textwidth`, inside a `figure*` on its own page. The
caption should state three things and no more: that the left column is behavioural and the right
representational, that the dashed verticals are that geometry's own predicted sites, and that the
figure is descriptive — it shows where frequency information is lost, not that the loss is
structured by the tokenisation geometry, which is what the Bayesian models decide.
